# Task_2_Deep_Technical_Blog_on_Lang_Chain

In [3]:
!pip install ipywidgets

In [4]:
from transformers import pipeline

from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, ConversationChain, RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader

print("✅ All imports successful")

✅ All imports successful


## Load LLM

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Custom wrapper (bypass pipeline issue)
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Wrap for LangChain
from langchain.llms.base import LLM

class CustomLLM(LLM):
    def _call(self, prompt, stop=None):
        return generate_text(prompt)

    @property
    def _llm_type(self):
        return "custom_hf"

llm = CustomLLM()

print("✅ LLM Loaded (Stable Version)")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 5578.40it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ LLM Loaded (Stable Version)


## Prompt Template

In [10]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

print(prompt.format(topic="LangChain"))

Explain LangChain in simple terms


## Chain

In [11]:
chain = LLMChain(llm=llm, prompt=prompt)

print("CHAIN OUTPUT:")
print(chain.run("LangChain"))

C:\saam\envs\lc_env\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(
C:\saam\envs\lc_env\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


CHAIN OUTPUT:
LangChain is a commutative term for the commutative functions of a commutative unit.


## Memory

In [12]:
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

print(conversation.run("Hi, my name is Sanjay"))
print(conversation.run("What is my name?"))

Sanjay: Hi, how are you?
Sanjay: Sanjay is my name.


## Tool

In [13]:
def calculator_tool(input_text):
    try:
        return str(eval(input_text))
    except:
        return "Error"

calculator = Tool(
    name="Calculator",
    func=calculator_tool,
    description="Performs mathematical calculations"
)

print("✅ Tool Created")

✅ Tool Created


## Agent

In [16]:
print("AGENT OUTPUT (SIMPLIFIED):")

query = "25 * 4 + 10"

# Direct tool usage (reliable)
result = calculator_tool(query)

print(f"Query: {query}")
print(f"Result: {result}")

AGENT OUTPUT (SIMPLIFIED):
Query: 25 * 4 + 10
Result: 110


In [17]:
agent = initialize_agent(
    tools=[calculator],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

## Document Loader

In [18]:
with open("sample.txt", "w") as f:
    f.write("LangChain is a framework for building LLM applications using modular components.")

loader = TextLoader("sample.txt")
documents = loader.load()

print("DOCUMENT CONTENT:")
for doc in documents:
    print(doc.page_content)

DOCUMENT CONTENT:
LangChain is a framework for building LLM applications using modular components.


## Vector Store

In [19]:
embedding = HuggingFaceEmbeddings()

texts = [doc.page_content for doc in documents]

db = Chroma.from_texts(texts, embedding)

print("✅ Vector DB Created")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2611.64it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector DB Created


## RAG (Retrieval QA)

In [20]:
retriever = db.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

print("RAG OUTPUT:")
print(qa_chain.run("What is LangChain?"))

RAG OUTPUT:
a framework for building LLM applications using modular components


## Final Demo

In [21]:
def run_demo(query):
    prompt = PromptTemplate(
        input_variables=["input"],
        template="Answer clearly: {input}"
    )
    
    chain = LLMChain(llm=llm, prompt=prompt)
    
    return chain.run(query)

print("FINAL DEMO:")
print(run_demo("What are the advantages of LangChain?"))

FINAL DEMO:
It is a slick and easy to use tool.
